# Tarea 2 — Parte 2: Evaluación y extracción estructurada
Muestra fija de **10 documentos** (3 impresos, 3 fotos, 4 manuscritos) y **10 clips** (2 multihablante + 8 de distintos hablantes), elegida con semilla antes de mirar métricas. Las configuraciones de todos los modelos se fijaron antes de calcular resultados y no se ajustaron después.

**Referencias manuales:** cada referencia está en `datos/referencias/<id>.txt`. Se partió de un borrador (capa de texto del PDF del BCRP, guion de lectura de OpenSLR o el texto copiado a mano en los manuscritos) y se **revisó leyendo la imagen / escuchando el audio completo**, corrigiendo omisiones y diferencias. Un archivo que aún tiene la línea `# BORRADOR` no se usa.

In [ ]:
import json, random, time, datetime
import numpy as np, pandas as pd
import t2lib as T
pd.set_option("display.max_colwidth", 100)
docs = pd.read_csv(T.DATOS / "documentos.csv").fillna({"texto_referencia": ""})
aud = pd.read_csv(T.DATOS / "transcripciones.csv").fillna({"texto_referencia": ""})
print(docs.shape, aud.shape)

In [ ]:

import plotly.express as px
def mostrar(fig):
    try: fig.show(renderer="notebook_connected+png")
    except Exception:
        try:
            import kaleido; kaleido.get_chrome_sync(); fig.show(renderer="notebook_connected+png")
        except Exception as e:
            print("(PNG no disponible:", e, ")"); fig.show(renderer="notebook_connected")


## 1. Muestra de evaluación (fija)

In [ ]:
mpath = T.DATOS / "muestra_eval.csv"
if mpath.exists():
    muestra = pd.read_csv(mpath)
else:
    rng = random.Random(2026)
    sel_d = []
    for tipo, k in [("impreso", 3), ("foto", 3), ("manuscrito", 4)]:
        ids = sorted(docs[docs.tipo == tipo].id); sel_d += rng.sample(ids, min(k, len(ids)))
    multi = sorted(aud[aud.n_hablantes_fuente > 1].id)[:2]
    sel_a = multi + rng.sample(sorted(set(aud.id) - set(multi)), 10 - len(multi))
    muestra = pd.DataFrame({"id": sel_d + sel_a, "modalidad": ["documento"] * len(sel_d) + ["audio"] * len(sel_a)})
    muestra.to_csv(mpath, index=False)
muestra.groupby("modalidad").id.apply(list)

In [ ]:
# Crea borradores de referencia (solo si no existen). La autora los revisa y borra la línea '# BORRADOR'.
import re
manus = re.split(r"^## (\d\d) — .*$", (T.REFS / "manuscritos_textos.md").read_text(encoding="utf-8"), flags=re.M)
manus = {int(manus[i]): manus[i + 1].strip() for i in range(1, len(manus), 2)}
d_idx, a_idx = docs.set_index("id"), aud.set_index("id")
fotos_orden = list(docs[docs.tipo == "foto"].id); manus_orden = list(docs[docs.tipo == "manuscrito"].id)
for _, m in muestra.iterrows():
    f = T.REFS / f"{m.id}.txt"
    if f.exists(): continue
    if m.modalidad == "audio":
        borr = a_idx.loc[m.id, "guion_openslr"]
    elif d_idx.loc[m.id, "tipo"] == "impreso":
        borr = (T.REFS / "borradores" / f"{m.id}_pdf.txt").read_text(encoding="utf-8")
    elif d_idx.loc[m.id, "tipo"] == "foto":
        g = T.REFS / "para_fotografiar" / f"foto_{fotos_orden.index(m.id) + 1:02d}_pdf.txt"
        borr = g.read_text(encoding="utf-8") if g.exists() else ""
    else:
        borr = manus.get(manus_orden.index(m.id) + 1, "")
    f.write_text("# BORRADOR — revisar contra la imagen/audio y borrar esta línea\n" + borr, encoding="utf-8")
pend = [m for m in muestra.id if (T.REFS / f"{m}.txt").read_text(encoding="utf-8").startswith("# BORRADOR")]
print("Referencias pendientes de revisión manual:", pend if pend else "ninguna ✓")

In [ ]:
def ref(i):
    t = (T.REFS / f"{i}.txt").read_text(encoding="utf-8")
    return "" if t.startswith("# BORRADOR") else t.strip()
docs["texto_referencia"] = [ref(i) if i in set(muestra.id) else "" for i in docs.id]
aud["texto_referencia"] = [ref(i) if i in set(muestra.id) else "" for i in aud.id]
docs.to_csv(T.DATOS / "documentos.csv", index=False); aud.to_csv(T.DATOS / "transcripciones.csv", index=False)
ev_d = docs[docs.id.isin(muestra.id) & (docs.texto_referencia != "")].copy()
ev_a = aud[aud.id.isin(muestra.id) & (aud.texto_referencia != "")].copy()
print(f"Referencias listas: {len(ev_d)}/10 documentos, {len(ev_a)}/10 clips")

## 2. Línea base sin ajuste
Normalización idéntica para referencia e hipótesis: minúsculas, NFC, sin puntuación, espacios colapsados y sin marcas `[HABLANTE_n]`.

In [ ]:
ev_d["cer_paddle"] = [T.cer(r, h) for r, h in zip(ev_d.texto_referencia, ev_d.texto_ocr)]
ev_a["wer_whisper"] = [T.wer(r, h) for r, h in zip(ev_a.texto_referencia, ev_a.texto_asr)]
print(f"BASE  PaddleOCR CER = {ev_d.cer_paddle.mean():.3f} | faster-whisper small WER = {ev_a.wer_whisper.mean():.3f}")
display(ev_d[["id", "tipo", "cer_paddle"]]); display(ev_a[["id", "n_hablantes_fuente", "wer_whisper", "rtf_asr"]])

## 3. Modelos alternativos sobre las mismas muestras
* **OCR:** EasyOCR 1.7 (`['es']`, CPU, `paragraph=True`) — detector CRAFT + reconocedor CRNN; arquitectura distinta a PaddleOCR y se instala solo con pip.
* **ASR:** `jonatasgrosman/wav2vec2-large-xlsr-53-spanish` (wav2vec2 XLS‑R 300M, CTC sin modelo de lenguaje) — familia distinta a Whisper.

Factor de tiempo real (RTF) = segundos de proceso / segundos de audio.

In [ ]:
cache = T.RAW / "eval_cache"; cache.mkdir(exist_ok=True)
def cacheado(nombre, fn):
    p = cache / f"{nombre}.json"
    if p.exists(): return json.loads(p.read_text(encoding="utf-8"))
    r = fn(); p.write_text(json.dumps(r, ensure_ascii=False), encoding="utf-8"); return r

alt = [cacheado(f"easy_{i}", lambda a=a: T.ocr_easyocr(T.BASE / a)) for i, a in zip(ev_d.id, ev_d.archivo)]
ev_d["texto_easyocr"] = [T.seudonimizar(t)[0] for t, _ in alt]
ev_d["seg_easyocr"] = [s for _, s in alt]
ev_d["cer_easyocr"] = [T.cer(r, h) for r, h in zip(ev_d.texto_referencia, ev_d.texto_easyocr)]

alt = [cacheado(f"w2v_{i}", lambda a=a: T.asr_wav2vec2(T.BASE / a)) for i, a in zip(ev_a.id, ev_a.archivo_audio)]
ev_a["texto_w2v"] = [T.seudonimizar(t)[0] for t, _ in alt]
ev_a["rtf_w2v"] = np.array([s for _, s in alt]) / ev_a.duracion_seg.values
ev_a["wer_w2v"] = [T.wer(r, h) for r, h in zip(ev_a.texto_referencia, ev_a.texto_w2v)]
display(ev_d[["id", "tipo", "cer_paddle", "cer_easyocr"]]); display(ev_a[["id", "wer_whisper", "wer_w2v", "rtf_asr", "rtf_w2v"]])

## 4. Extracción estructurada con LLM
Proveedor por variables de entorno: `LLM_PROVEEDOR=gemini` + `GEMINI_API_KEY` (por defecto `gemini-2.5-flash`), o `LLM_PROVEEDOR=ollama` + `LLM_MODELO=qwen2.5:3b`. Temperatura 0, salida JSON; `null` para datos ausentes.

In [ ]:
llm_dir = T.RAW / "llm_cache"; llm_dir.mkdir(exist_ok=True)
prov, modelo = T.llm_proveedor()
filas = []
for mod, df, col in [("documento", docs, "texto_ocr"), ("audio", aud, "texto_asr")]:
    for _, r in df.iterrows():
        p = llm_dir / f"{r.id}.json"
        if not p.exists():
            tipo = r.get("tipo", "entrevista/clip de audio") if mod == "documento" else "transcripción de audio"
            for intento in range(3):
                try:
                    p.write_text(json.dumps(T.extraer_llm(r[col], mod, tipo), ensure_ascii=False), encoding="utf-8"); break
                except Exception as e:
                    print(r.id, "reintento por:", str(e)[:80]); time.sleep(15)
            time.sleep(4 if prov == "gemini" else 0)   # límite de peticiones por minuto
        if p.exists():
            filas.append({"id": r.id, "modalidad": mod, "campos_json": p.read_text(encoding="utf-8"),
                          "modelo_llm": f"{prov}:{modelo}", "fecha": datetime.date.today().isoformat()})
llm = pd.DataFrame(filas); llm.to_csv(T.DATOS / "extraccion_llm.csv", index=False)
print(llm.shape); llm.head()

## 5. Auditoría humana de 20 extracciones al azar
Criterio: una extracción es **completamente correcta** si `tema` y `resumen` son fieles al texto, ningún campo contiene datos que no estén en el texto (sin invenciones), y los campos presentes en el texto (`emisor`, `fecha`, `monto` / `numero_hablantes`, `citas_clave`) no quedaron en `null`. Se completa a mano la columna `correcto` (1/0) y `error` en `datos/auditoria_llm.csv`.

In [ ]:
apath = T.DATOS / "auditoria_llm.csv"
if not apath.exists():
    rng = random.Random(7)
    ids = rng.sample(list(llm[llm.modalidad == "documento"].id), 10) + rng.sample(list(llm[llm.modalidad == "audio"].id), 10)
    textos = pd.concat([docs.set_index("id").texto_ocr, aud.set_index("id").texto_asr])
    a = llm.set_index("id").loc[ids, ["modalidad", "campos_json"]].reset_index()
    a["texto_fuente"] = a.id.map(textos).str[:1500]
    a["correcto"] = ""; a["error"] = ""
    a.to_csv(apath, index=False)
aud_llm = pd.read_csv(apath)
hechos = pd.to_numeric(aud_llm.correcto, errors="coerce")
print(f"Auditadas: {hechos.notna().sum()}/20")
if hechos.notna().sum() == 20:
    ok = int(hechos.sum()); print(f"Acuerdo LLM‑Humano: {ok}/20 = {ok/20:.0%}")
    display(aud_llm[hechos == 0][["id", "modalidad", "error"]])

## 6. Casos difíciles

In [ ]:
por_tipo = ev_d.groupby("tipo")[["cer_paddle", "cer_easyocr"]].mean().round(3)
display(por_tipo)
ev_a["grupo"] = np.where(ev_a.n_hablantes_fuente > 1, "multihablante", "un hablante")
por_grupo = ev_a.groupby("grupo")[["wer_whisper", "wer_w2v"]].mean().round(3)
display(por_grupo)
larga = pd.concat([
    por_tipo.reset_index().melt("tipo", var_name="modelo", value_name="error").rename(columns={"tipo": "caso"}),
    por_grupo.reset_index().melt("grupo", var_name="modelo", value_name="error").rename(columns={"grupo": "caso"})])
peor = larga.loc[larga.error.idxmax()]
fig = px.bar(larga, x="caso", y="error", color="modelo", barmode="group",
             title=f"El peor caso es '{peor.caso}' ({peor.modelo}: {peor.error:.2f}); los impresos son el caso más fácil")
mostrar(fig)

## 7. Tabla comparativa y decisión

In [ ]:
tabla = pd.DataFrame([
    {"modalidad": "documento", "modelo": "PaddleOCR (base)", "métrica": "CER", "valor": ev_d.cer_paddle.mean(), "seg_por_item": docs.seg_ocr.mean()},
    {"modalidad": "documento", "modelo": "EasyOCR (alternativo)", "métrica": "CER", "valor": ev_d.cer_easyocr.mean(), "seg_por_item": ev_d.seg_easyocr.mean()},
    {"modalidad": "audio", "modelo": "faster-whisper small (base)", "métrica": "WER", "valor": ev_a.wer_whisper.mean(), "rtf": ev_a.rtf_asr.mean()},
    {"modalidad": "audio", "modelo": "wav2vec2 XLS-R es (alternativo)", "métrica": "WER", "valor": ev_a.wer_w2v.mean(), "rtf": ev_a.rtf_w2v.mean()},
]).round(3)
display(tabla)
tabla.to_csv(T.DATOS / "metricas.csv", index=False)
det = pd.concat([ev_d[["id", "tipo", "cer_paddle", "cer_easyocr", "texto_easyocr"]].assign(modalidad="documento"),
                 ev_a[["id", "grupo", "wer_whisper", "wer_w2v", "texto_w2v"]].assign(modalidad="audio")])
det.to_csv(T.DATOS / "eval_detalle.csv", index=False)
mejor_ocr = tabla[tabla.modalidad == "documento"].nsmallest(1, "valor").modelo.item()
mejor_asr = tabla[tabla.modalidad == "audio"].nsmallest(1, "valor").modelo.item()
print("Menor CER:", mejor_ocr, "| Menor WER:", mejor_asr)

**Decisión (completar tras ver la tabla):** para un entorno real elegiría … para documentos porque … (precisión) y … (costo/latencia en CPU). Para audio elegiría … porque … ; wav2vec2 no usa modelo de lenguaje ni puntuación, por lo que …

**Reflexión por tipo:** el peor desempeño OCR se dio en … (causa: trazo irregular / sombra e inclinación / …). En audio, los clips multihablante … porque …

## 8. Resultado para el comentario del Issue

In [ ]:
acuerdo = f"{int(hechos.sum())}/20" if hechos.notna().sum() == 20 else "pendiente"
w = tabla.query("modelo == @mejor_asr").valor.item(); c = tabla.query("modelo == @mejor_ocr").valor.item()
linea = f"Resultados Test: WER: {w:.2f} | CER: {c:.2f} | Acuerdo LLM-Humano: {acuerdo}"
print(linea)
(T.DATOS / "resumen_test.txt").write_text(linea, encoding="utf-8")